# 🚨 Financial Red Flag Detector — Training Pipeline

**Run this notebook on Kaggle (CPU or GPU, internet ON)**

What this does:
1. Install missing dependencies
2. Clone our repo / set up paths
3. Fetch S&P 1500 data from SEC EDGAR (~25 min first run)
4. Engineer 46 features
5. Scrape AAER fraud labels
6. Train Fraud Model + Distress Model
7. Walk-forward cross-validation
8. Benchmark vs Altman Z + Beneish M
9. Save model artifacts

**After running:** Download `fraud_model.pkl` and `distress_model.pkl` from the output panel → commit to `models/` in the GitHub repo.

## Cell 1 — Install dependencies

In [ ]:
# Kaggle has most packages pre-installed.
# Only install what's missing.
!pip install -q rapidfuzz imbalanced-learn shap beautifulsoup4

# ── Verify GPU ───────────────────────────────────────────────────────────────
import subprocess, torch

gpu_available = torch.cuda.is_available()
if gpu_available:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'🟢 GPU detected: {result.stdout.strip()}')
    print('   XGBoost will use CUDA — expect ~10x speedup over CPU')
else:
    print('🟡 No GPU — running on CPU')
    print('   Tip: Settings → Accelerator → GPU T4 x1 for faster training')

print('\n✅ Dependencies ready')

## Cell 2 — Clone repo and configure paths

In [ ]:
import os, sys
from pathlib import Path

# Clone the repo into /kaggle/working/
REPO_URL = 'https://github.com/ananyashriram10/financial-red-flag-detector.git'
REPO_DIR = Path('/kaggle/working/financial-red-flag-detector')

if not REPO_DIR.exists():
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print(f'Cloned → {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')
    print(f'Updated → {REPO_DIR}')

# Add to Python path
sys.path.insert(0, str(REPO_DIR))

# Create data dirs
(REPO_DIR / 'data' / 'raw').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'data' / 'labels').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'data' / 'processed').mkdir(parents=True, exist_ok=True)

print('✅ Paths configured')
print(f'Working dir: {REPO_DIR}')

## Cell 3 — Fetch S&P 1500 from SEC EDGAR

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

from pipeline.edgar_bulk import build_raw_table, get_sp1500_tickers

# Get ticker universe
tickers = get_sp1500_tickers()
print(f'Universe: {len(tickers)} tickers')

# Fetch EDGAR data (cached after first run — re-runs take < 1 min)
raw_df = build_raw_table(tickers=tickers)

print(f'\nRaw financials shape: {raw_df.shape}')
print(f'Companies: {raw_df["cik"].nunique()}')
print(f'Year range: {raw_df["year"].min()} – {raw_df["year"].max()}')
raw_df.head()

## Cell 4 — Engineer 46 features

In [ ]:
from pipeline.features import engineer_features
import pandas as pd

feat_df = engineer_features(raw_df)

feat_cols = [c for c in feat_df.columns if c.startswith('f_')]
print(f'Features: {len(feat_cols)}')
print(f'Shape: {feat_df.shape}')
print(f'\nFeature completeness (non-null %)')
print((feat_df[feat_cols].notna().mean() * 100).sort_values().tail(20).to_string())

# Save
feat_path = REPO_DIR / 'data' / 'processed' / 'features.parquet'
feat_df.to_parquet(feat_path, index=False)
print(f'\n✅ Features saved → {feat_path}')

## Cell 5 — Scrape AAER fraud labels

In [ ]:
## Cell 4b — Augment dataset with fraud company EDGAR data
#
# Problem: fraud companies (Nikola, MiMedx, old Enron/WorldCom) are often
# NOT in the S&P 500/1500 universe, so their features are missing.
# Without features, fraud labels can't attach to any rows → fraud rate = 0%.
#
# Fix: resolve fraud company CIKs first, then fetch their EDGAR data
# and add it to raw_df before feature engineering.

from pipeline.aaer_labels import get_seed_fraud_labels, KNOWN_FRAUD_SEED
from pipeline.edgar_bulk import augment_with_fraud_companies

# Step 1: resolve fraud company CIKs from seed list
print("Resolving fraud company CIKs...")
seed_df = get_seed_fraud_labels()
print(f"  Resolved: {len(seed_df)} companies")
print(seed_df[['ticker', 'company_name', 'fraud_date']].to_string())

# Step 2: fetch EDGAR data for any that are missing from our dataset
fraud_ciks = seed_df['cik'].astype(str).tolist()
raw_df = augment_with_fraud_companies(raw_df, fraud_ciks, min_years=2)

print(f"\nDataset after augmentation: {raw_df['cik'].nunique()} companies, {len(raw_df):,} rows")

# Step 3: re-run feature engineering on augmented dataset
from pipeline.features import engineer_features
feat_df = engineer_features(raw_df)
feat_path = REPO_DIR / 'data' / 'processed' / 'features.parquet'
feat_df.to_parquet(feat_path, index=False)
print(f"Features re-saved with augmented data → {feat_path}")
print(f"Shape: {feat_df.shape}")

In [ ]:
from pipeline.aaer_labels import build_fraud_labels, load_supplemental_aaer_dataset

# Scrape SEC AAER releases (cached to data/labels/fraud_labels.csv)
fraud_df = build_fraud_labels(use_cache=False)
print(f'AAER fraud companies found: {len(fraud_df)}')

# Also try academic dataset (Dechow et al. 2011)
supp = load_supplemental_aaer_dataset()
if not supp.empty:
    print(f'Supplemental academic labels: {len(supp)} rows')

print('\nSample fraud labels:')
fraud_df[['ticker', 'company_name', 'fraud_date', 'aaer_no', 'match_score']].head(20)

## Cell 6 — Assemble labeled dataset

In [ ]:
from pipeline.labeler import build_labeled_dataset

labeled_df = build_labeled_dataset(
    features_path=REPO_DIR / 'data' / 'processed' / 'features.parquet',
    fraud_labels_path=REPO_DIR / 'data' / 'labels' / 'fraud_labels.csv',
)

print(f'\nFinal dataset shape: {labeled_df.shape}')
print(f'\nLabel distribution:')
print(labeled_df['label_type'].value_counts())
print(f'\nFraud rate:    {labeled_df["is_fraud"].mean():.2%}')
print(f'Bankrupt rate: {labeled_df["is_bankrupt"].mean():.2%}')

labeled_df[['ticker', 'year', 'label_type', 'is_fraud', 'is_bankrupt']].head(20)

## Cell 7 — Walk-forward CV (evaluate before final training)

In [ ]:
from models.fraud_model import FraudModel
from models.distress_model import DistressModel

fraud_model    = FraudModel()
distress_model = DistressModel()

print('Running walk-forward CV on Fraud Model...')
fraud_cv = fraud_model.walk_forward_evaluate(
    labeled_df,
    min_train_years=8,
    test_window=3,
    step=3,
)

print('\nRunning walk-forward CV on Distress Model...')
distress_cv = distress_model.walk_forward_evaluate(
    labeled_df,
    min_train_years=8,
    test_window=3,
    step=3,
)

def print_cv_results(name, cv):
    folds = cv.get('fold_metrics', None)
    print(f'\n=== {name} — CV Results ===')
    if folds is None or folds.empty:
        print('  No valid folds — not enough positive labels in training windows.')
        print('  This usually means fraud labels cover too few companies or years.')
        print(f'  OOF AUC-ROC: {cv.get("oof_auc_roc", "N/A")}')
    else:
        cols = [c for c in ['fold','val_years','n_fraud_val','auc_roc','auc_pr']
                if c in folds.columns]
        print(folds[cols].to_string(index=False))
        print(f'OOF AUC-ROC: {cv.get("oof_auc_roc")}  OOF AUC-PR: {cv.get("oof_auc_pr")}')

print_cv_results('Fraud Model',   fraud_cv)
print_cv_results('Distress Model', distress_cv)

## Cell 8 — Train final production models on full dataset

In [ ]:
print('Training final Fraud Model on full dataset...')
fraud_model.fit(labeled_df)

print('\nTraining final Distress Model on full dataset...')
distress_model.fit(labeled_df)

print('✅ Both models trained')

## Cell 9 — Benchmark vs Altman Z + Beneish M

In [ ]:
from models.evaluate import run_evaluation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

results = run_evaluation(
    labeled_df,
    fraud_model,
    distress_model,
    test_year_cutoff=2016,
)

# ── Plot: AUC-ROC comparison bar chart ───────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Fraud Detection', 'Distress Detection'])

colors = {'Altman Z-Score': '#6B7280', 'Beneish M-Score': '#D97706',
          'Our Model (XGB + IsoForest)': '#2563EB'}

for task, col in [('fraud', 1), ('distress', 2)]:
    bench = results.get(f'{task}_benchmark', [])
    if not bench:
        continue
    models_  = [r['model'] for r in bench]
    auc_rocs = [r['auc_roc'] for r in bench]
    fig.add_trace(
        go.Bar(x=models_, y=auc_rocs,
               marker_color=[colors.get(m, '#94A3B8') for m in models_],
               text=[f'{v:.3f}' for v in auc_rocs], textposition='outside',
               showlegend=False),
        row=1, col=col
    )

fig.update_layout(
    title='AUC-ROC: Our Model vs Altman Z vs Beneish M',
    paper_bgcolor='#0F172A', plot_bgcolor='#1E293B',
    font=dict(color='#F1F5F9'),
    height=400,
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

In [ ]:
# ── Plot: Lift curves ────────────────────────────────────────────────────────
import pandas as pd

lift_df = pd.DataFrame(results['fraud_lift'])

fig2 = px.line(
    lift_df, x='pct_reviewed', y='recall',
    color='model', title='Lift Curve — Fraud Detection',
    labels={'pct_reviewed': '% Companies Reviewed (top-K%)',
            'recall': '% Frauds Caught'},
    color_discrete_map=colors,
    template='plotly_dark',
)
# Random baseline
fig2.add_trace(go.Scatter(
    x=[0,100], y=[0,1], name='Random Baseline',
    line=dict(dash='dash', color='#475569')
))
fig2.update_layout(height=450)
fig2.show()

print('\nKey insight: At 5% review rate, our model catches what % of frauds?')
for m in lift_df['model'].unique():
    recall_5 = lift_df[(lift_df['model']==m) & (lift_df['pct_reviewed']==5.0)]['recall'].values
    if len(recall_5):
        print(f'  {m}: {recall_5[0]:.1%}')

In [ ]:
# ── Feature importance: what drives the fraud score? ────────────────────────
import pandas as pd

fi = pd.DataFrame(results['fraud_feature_importance'])
print('Top 20 features driving FRAUD predictions (mean |SHAP|):')
print(fi[['rank','label','mean_shap']].to_string(index=False))

## Cell 10 — Save model artifacts

In [ ]:
from pathlib import Path

out_dir = Path('/kaggle/working')

# Save models
fraud_path   = fraud_model.save(out_dir / 'fraud_model.pkl')
distress_path= distress_model.save(out_dir / 'distress_model.pkl')

# Save eval results
import json
with open(out_dir / 'eval_results.json', 'w') as f:
    import numpy as np
    def _safe(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        return str(o)
    json.dump(results, f, indent=2, default=_safe)

print('\n✅ Artifacts saved to /kaggle/working/')
print(f'  fraud_model.pkl    ({fraud_path.stat().st_size / 1e6:.1f} MB)')
print(f'  distress_model.pkl ({distress_path.stat().st_size / 1e6:.1f} MB)')
print(f'  eval_results.json')
print()
print('Next steps:')
print('  1. Download fraud_model.pkl + distress_model.pkl from the Output panel')
print('  2. Place them in models/ in the GitHub repo')
print('  3. git add models/*.pkl && git commit && git push')
print('  4. Deploy app on Streamlit Cloud → point to your repo')